# 01 解压原始数据与初步识别

本 Notebook 完成以下任务：
1. 自动创建项目目录结构
2. 解压 7 个 CSMAR 原始 ZIP 文件
3. 读取各文件并识别字段含义
4. 输出变量字典

In [ ]:
import os
import zipfile
import pandas as pd

# 1. 自动创建目录结构
BASE = os.path.dirname(os.path.abspath('__file__'))
dirs = [
    'data/data_raw_zip', 'data/raw', 'data/dict',
    'data/clean', 'data/combined', 'data/temp',
    'output/tables', 'output/figures'
]
for d in dirs:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
    print(f'Created: {d}')
print('目录结构创建完成')

## 1.2 解压原始 ZIP 文件

将 `data/data_raw_zip/` 下所有 ZIP 文件自动解压到 `data/raw/` 对应子目录。

In [ ]:
import zipfile
import os

zip_dir = 'data/data_raw_zip'
raw_dir = 'data/raw'

# 文件名到目录名的映射
zip_to_dir = {
    'CSMAR常用变量-2000-2024.zip': 'csmar_common_vars',
    '上市公司基本信息变更表2000-2024.zip': 'firm_info_change',
    '上市公司基本信息年度表.zip': 'firm_info_annual',
    '利润表-现金流量表-2000-2010.zip': 'income_cashflow_2000_2010',
    '利润表-现金流量表-2011-2024.zip': 'income_cashflow_2011_2024',
    '资产负债表-2000-2010.zip': 'balance_sheet_2000_2010',
    '资产负债表-2011-2024.zip': 'balance_sheet_2011_2024',
}

for fname, outdir in zip_to_dir.items():
    zip_path = os.path.join(zip_dir, fname)
    out_path = os.path.join(raw_dir, outdir)
    if os.path.exists(zip_path):
        os.makedirs(out_path, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(out_path)
        print(f'Extracted: {fname} -> {outdir}')
    else:
        print(f'NOT FOUND: {zip_path}')

## 1.3 识别各文件字段

读取每个原始文件的列名和前几行，建立对变量含义的初步认识。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def read_csmar_excel(path):
    """读取CSMAR格式Excel，跳过单位行（第2行）"""
    return pd.read_excel(path, header=0, skiprows=[1])

files = {
    '资产负债表_2000-2010': 'data/raw/balance_sheet_2000_2010/跨表查询_沪深京股票(年频).xlsx',
    '资产负债表_2011-2024': 'data/raw/balance_sheet_2011_2024/跨表查询_沪深京股票(年频).xlsx',
    '利润表/现金流量表_2000-2010': 'data/raw/income_cashflow_2000_2010/跨表查询_沪深京股票(年频).xlsx',
    '利润表/现金流量表_2011-2024': 'data/raw/income_cashflow_2011_2024/跨表查询_沪深京股票(年频).xlsx',
    'CSMAR常用变量': 'data/raw/csmar_common_vars/常用变量查询（年度）.xlsx',
    '公司基本信息年度表': 'data/raw/firm_info_annual/STK_LISTEDCOINFOANL.xlsx',
}

for name, path in files.items():
    df = read_csmar_excel(path)
    print(f'\n=== {name} ===')
    print(f'行数: {len(df)}, 列数: {len(df.columns)}')
    print('列名:', df.columns.tolist()[:8], '...')

## 1.4 输出变量字典

读取已生成的 `variable_dictionary.csv`，确认变量来源映射。

In [ ]:
var_dict = pd.read_csv('data/dict/variable_dictionary.csv', encoding='utf-8-sig')
print(f'变量字典: {len(var_dict)} 条记录')
var_dict.head(10)